Recreacion del codigo para ejecutar el algoritmo txmeans, puediendolo depurar de una manera mas eficiente

In [1]:
import sys
sys.path.insert(0, r'/home/adrian/Escritorio/TFG/TXMeans/code')

In [2]:
from algorithms.txmeans import *
from generators.datamanager import *
from validation.validation_measures import *
from generators.datagenerator import *


Leer y modificar el dataset

In [3]:
path = '../../dataset/'
dataset_name = 'mushrooms.csv'

txmeans = TXmeans()
    
filename = path + dataset_name
class_index = 0
skipcolumnsindex = set()
    
baskets_real_labels, maps = read_uci_data(filename, class_index=class_index, skipcolumnsindex=skipcolumnsindex)

print( dataset_name, len(baskets_real_labels))

mushrooms.csv 8124


/home/adrian/Escritorio/TFG/TXMeans/code/generators/datamanager.py:39: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode_value = mode(df[k])[0][0]


Preparar el dataset para ser ejecutado, añadiendo las transacciones en baskets, modificandolas a bits

In [4]:
baskets_list = list()
real_labels = list()
count = 0
for basket, label in baskets_real_labels:
    baskets_list.append(basket)
    real_labels.append(label)
    count += 1
baskets_list, map_newitem_item, map_item_newitem = remap_items(baskets_list)
baskets_list = basket_list_to_bitarray(baskets_list, len(map_newitem_item))

nbaskets = len(baskets_list)
nitems = count_items(baskets_list)

Ejecucion del algoritmo txmeans

In [5]:
start_time = datetime.datetime.now()

nsample = sample_size(nbaskets, 0.05, conf_level=0.99, prob=0.5)
txmeans.fit(baskets_list, nbaskets, nitems, random_sample=nsample)


end_time = datetime.datetime.now()
running_time = end_time - start_time

Resultados Pd cada vez que se ejecuta el algoritmo los resultasdos varian 

In [6]:
res = txmeans.clustering
#iter_count = bicartd.iter_count
pred_labels = [0] * len(real_labels)
baskets_clusters = list()
for cluster, label in zip(res, range(0, len(res))):
    cluster_list = basket_bitarray_to_list(cluster['cluster']).values()
    for bid in cluster['cluster']:
        pred_labels[bid] = label
        baskets_clusters.append(cluster_list)

print('delta_k', delta_k(real_labels, pred_labels))
print('normalized_mutual_info_score', normalized_mutual_info_score(real_labels, pred_labels))
print('purity', purity(real_labels, pred_labels))
print('running_time', running_time)

delta_k 3
normalized_mutual_info_score 0.35213904476629476
purity 0.8906942392909897
running_time 0:00:00.099383
